# JupyterLite で学ぶ Mesa 入門：エージェントベースモデル（ABM）

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
Python のエージェントベースモデル（ABM）ライブラリ **Mesa** の基礎を一から学ぶためのチュートリアルです。

## 対象者
- Python の基本（変数、リスト、関数、クラス）を理解している方
- 「個々の主体（エージェント）の行動から市場や社会の動きを再現する」シミュレーションに興味がある方
- Mesa を初めて使う方

## このチュートリアルで学ぶこと
1. エージェントベースモデル（ABM）とは
2. エージェントとモデルの定義、ステップ実行
3. 富の分配モデル（Boltzmann wealth model）とジニ係数
4. 格子空間（MultiGrid）上の移動と近傍
5. 買い手と売り手の市場モデル
6. ネットワーク上の情報伝播（口コミ）
7. パラメータ実験と再現性

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 各章の最後に **練習問題** があります。「解答欄」に自分でコードを書いてから「解答例」を開いて確認しましょう。
- このノートブックは **Mesa 3 系** の書き方（`Agent(model)`、`model.agents`）で統一しています。Mesa 2 系の `RandomActivation` は使いません。

## 0. 環境準備（JupyterLite 用）

まず、必要なライブラリをインストールします（初回は数十秒かかることがあります）。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["scipy", "mesa", "networkx", "numpy", "pandas", "matplotlib", "japanize-matplotlib-jlite"])
except ImportError:
    pass

import scipy  # networkx の一部の関数（レイアウト・中心性・PageRank）が内部で使うため先に読み込む


In [ ]:
import mesa
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語表示用（plt の後に import する）
from mesa.datacollection import DataCollector
from mesa.space import MultiGrid

plt.rcParams["figure.figsize"] = (8, 4.5)
print(f"Mesa バージョン: {mesa.__version__}")
print(f"NetworkX バージョン: {nx.__version__}")

---
## 1. エージェントベースモデル（ABM）とは

**エージェントベースモデル（Agent-Based Model, ABM）** は、多数の主体（**エージェント**）が
それぞれ簡単なルールに従って行動し、相互作用することで、全体としてどのような現象（**創発**）が
生じるかをコンピュータ上で再現する手法です。

経済学では、次のような問題に使われます。

| テーマ | エージェント | 見たい現象 |
|---|---|---|
| 所得・資産の分布 | 家計 | ジニ係数、格差の拡大 |
| 市場の価格形成 | 買い手・売り手 | 均衡価格への収束、バブル |
| 住み分け（Schelling モデル） | 住民 | 少しの選好から生じる大きな分離 |
| 新製品の普及 | 消費者 | S 字型の普及曲線、口コミ効果 |
| 金融危機 | 銀行 | 連鎖的な破綻 |

### Mesa の構成要素

| 部品 | 役割 |
|---|---|
| `mesa.Agent` | 1 体のエージェント。`step()` に毎期の行動を書く |
| `mesa.Model` | シミュレーション全体。エージェントの生成、`step()` で 1 期進める |
| `model.agents` | モデル内の全エージェント（`AgentSet`）。`shuffle_do("step")` でランダム順に行動させる |
| `DataCollector` | 毎期の値（モデル全体・各エージェント）を記録し、DataFrame に変換する |
| `MultiGrid` など | 空間。エージェントの位置と近傍を管理する |

---
## 2. 最初のモデル：エージェントとモデルの定義

### 2.1 エージェントとモデルを書く

Mesa 3 では、エージェントは `mesa.Agent` を継承し、`__init__(self, model)` で `super().__init__(model)` を呼びます。
モデルは `mesa.Model` を継承し、`super().__init__(rng=seed)` で乱数の種を渡せます。

In [ ]:
class HelloAgent(mesa.Agent):
    """挨拶するだけのエージェント"""

    def __init__(self, model):
        super().__init__(model)          # 必ず呼ぶ（unique_id が自動で付く）

    def step(self):
        print(f"こんにちは、私はエージェント {self.unique_id} です")


class HelloModel(mesa.Model):
    """エージェントを n 体つくるだけのモデル"""

    def __init__(self, n, seed=None):
        super().__init__(rng=seed)      # rng に種（整数）を渡すと結果を再現できる
        for _ in range(n):
            HelloAgent(self)             # 生成すると自動的に model.agents に登録される

    def step(self):
        self.agents.shuffle_do("step")   # 全エージェントをランダムな順番で step() させる


model = HelloModel(3, seed=1)
print("エージェント数:", len(model.agents))

### 2.2 モデルを動かす

`model.step()` を呼ぶたびに 1 期進みます。`model.steps` に経過した期数が自動で記録されます。

In [ ]:
for _ in range(2):
    print(f"--- 第 {model.steps + 1} 期 ---")
    model.step()

print("経過した期数:", model.steps)

### 2.3 AgentSet の便利な操作

`model.agents` は **AgentSet** というコレクションで、条件による絞り込みや属性の一括取得ができます。

In [ ]:
class WorkerAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.wage = self.random.randint(200, 400)   # self.random はモデルと共有された乱数生成器


class WorkerModel(mesa.Model):
    def __init__(self, n, seed=None):
        super().__init__(rng=seed)
        for _ in range(n):
            WorkerAgent(self)

    def step(self):
        pass


wm = WorkerModel(8, seed=3)
print("全員の賃金:", wm.agents.get("wage"))                       # 属性を一括取得
high = wm.agents.select(lambda a: a.wage >= 300)                  # 条件で絞り込み
print("300 以上の人数:", len(high))
print("平均賃金:", np.mean(wm.agents.get("wage")))
print("最初のエージェント:", wm.agents[0].unique_id, wm.agents[0].wage)

### 2.4 エージェントの追加と削除

シミュレーションの途中でエージェントが増えたり（企業の参入）、減ったり（退出）することもあります。
生成すれば自動で登録され、`agent.remove()` でモデルから外れます。

In [ ]:
wm2 = WorkerModel(5, seed=1)
print("最初の人数:", len(wm2.agents))

low = wm2.agents.select(lambda a: a.wage < 250)   # 賃金 250 未満の人を選ぶ
for a in list(low):
    a.remove()                                     # モデルから削除
print("250 未満を除いた人数:", len(wm2.agents))

WorkerAgent(wm2)                                   # 新しい人が参加
print("1 人追加した人数:", len(wm2.agents), " 賃金一覧:", wm2.agents.get("wage"))

### 練習問題 1

1. `step()` のたびに `age` が 1 増える `PersonAgent` と、それを 5 体持つ `PersonModel` を作り、3 期進めて全員の `age` を表示してください。
2. `PersonModel` に `seed=10` を渡し、`__init__` で `age` の初期値を `self.random.randint(18, 60)` にしてください。3 期後に 40 歳以上のエージェントの数を `select` で数えてください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
class PersonAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.age = self.random.randint(18, 60)

    def step(self):
        self.age += 1


class PersonModel(mesa.Model):
    def __init__(self, n, seed=None):
        super().__init__(rng=seed)
        for _ in range(n):
            PersonAgent(self)

    def step(self):
        self.agents.shuffle_do("step")


pm = PersonModel(5, seed=10)
for _ in range(3):
    pm.step()
print(pm.agents.get("age"))
print("40 歳以上:", len(pm.agents.select(lambda a: a.age >= 40)))
```

</details>

---
## 3. 富の分配モデル（Boltzmann wealth model）

ABM の定番モデルです。ルールはたった 1 つ。

> 各エージェントは毎期、お金を 1 単位以上持っていれば、ランダムに選んだ相手に 1 単位渡す。

全員が同じ 1 単位から始めても、繰り返すうちに **富の分布が大きく偏る** ことが知られています。

### 3.1 モデルの定義

In [ ]:
class MoneyAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.wealth = 1

    def step(self):
        if self.wealth > 0:
            other = self.random.choice(list(self.model.agents))   # ランダムに相手を選ぶ
            other.wealth += 1
            self.wealth -= 1


class MoneyModel(mesa.Model):
    def __init__(self, n, seed=None):
        super().__init__(rng=seed)
        for _ in range(n):
            MoneyAgent(self)

    def step(self):
        self.agents.shuffle_do("step")


money_model = MoneyModel(50, seed=42)
for _ in range(50):
    money_model.step()

wealth = money_model.agents.get("wealth")
print("最終期の富:", sorted(wealth, reverse=True))

### 3.2 分布をヒストグラムで見る

In [ ]:
plt.hist(wealth, bins=range(0, max(wealth) + 2), edgecolor="black")
plt.title("50 期後の富の分布（50 人）")
plt.xlabel("富（単位）")
plt.ylabel("人数")
plt.show()

### 3.3 ジニ係数と DataCollector

格差の大きさは **ジニ係数**（0 = 完全平等、1 = 1 人がすべてを所有）で測れます。
毎期の値を記録するには **`DataCollector`** を使います。

- `model_reporters`：モデル全体の値（関数にモデルを渡して計算）
- `agent_reporters`：各エージェントの属性（属性名を文字列で指定）

In [ ]:
def compute_gini(model):
    """モデル内の富のジニ係数を計算する"""
    x = sorted(model.agents.get("wealth"))
    n = len(x)
    total = sum(x)
    if total == 0:
        return 0.0
    cumulative = sum((i + 1) * xi for i, xi in enumerate(x))
    return (2 * cumulative) / (n * total) - (n + 1) / n


class MoneyModel2(mesa.Model):
    def __init__(self, n, seed=None):
        super().__init__(rng=seed)
        for _ in range(n):
            MoneyAgent(self)
        self.datacollector = DataCollector(
            model_reporters={"Gini": compute_gini},
            agent_reporters={"Wealth": "wealth"},
        )

    def step(self):
        self.datacollector.collect(self)      # 行動する前の状態を記録
        self.agents.shuffle_do("step")


money_model2 = MoneyModel2(50, seed=42)
for _ in range(100):
    money_model2.step()

gini_df = money_model2.datacollector.get_model_vars_dataframe()
print(gini_df.head())
print("最終期のジニ係数:", round(gini_df["Gini"].iloc[-1], 3))

### 3.4 推移をプロットする

In [ ]:
gini_df["Gini"].plot()
plt.title("ジニ係数の推移")
plt.xlabel("期")
plt.ylabel("ジニ係数")
plt.grid(True)
plt.show()

In [ ]:
# 各エージェントの富（Step と AgentID の 2 段階インデックス）
agent_df = money_model2.datacollector.get_agent_vars_dataframe()
print(agent_df.head())

# 最終期の富の分布
last_step = agent_df.index.get_level_values("Step").max()
final_wealth = agent_df.xs(last_step, level="Step")["Wealth"]
print("最終期:", last_step, "  最大:", final_wealth.max(), "  富 0 の人数:", (final_wealth == 0).sum())

In [ ]:
# 特定のエージェントの富の推移
agent_df.xs(1, level="AgentID")["Wealth"].plot(label="エージェント 1")
agent_df.xs(2, level="AgentID")["Wealth"].plot(label="エージェント 2")
plt.title("エージェント別の富の推移")
plt.xlabel("期")
plt.ylabel("富")
plt.legend()
plt.show()

### 3.5 ローレンツ曲線

ジニ係数の元になる **ローレンツ曲線**（人口の累積割合に対する富の累積割合）を描いてみます。
45 度線から離れるほど不平等です。

In [ ]:
w_sorted = np.sort(np.array(money_model2.agents.get("wealth"), dtype=float))
cum_people = np.arange(1, len(w_sorted) + 1) / len(w_sorted)
cum_wealth = np.cumsum(w_sorted) / w_sorted.sum()

plt.plot(cum_people, cum_wealth, marker=".", label="ローレンツ曲線")
plt.plot([0, 1], [0, 1], "k--", label="完全平等線")
plt.fill_between(cum_people, cum_wealth, cum_people, alpha=0.2)
plt.title(f"ローレンツ曲線（ジニ係数 {compute_gini(money_model2):.3f}）")
plt.xlabel("人口の累積割合")
plt.ylabel("富の累積割合")
plt.legend()
plt.show()

### 練習問題 2

1. `MoneyModel2` を 100 人・`seed=7` で 200 期動かし、最終期のジニ係数と、富が 0 の人数を表示してください。
2. 「毎期、富を 1 渡すのではなく、自分の富の半分（切り捨て）を渡す」ルールに変えた `HalfAgent` を作り、同じ条件でジニ係数の推移を元のモデルと比べてください（`DataCollector` の `model_reporters` は同じものを使えます）。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
m1 = MoneyModel2(100, seed=7)
for _ in range(200):
    m1.step()
df1 = m1.datacollector.get_model_vars_dataframe()
w1 = m1.agents.get("wealth")
print("ジニ係数:", round(df1["Gini"].iloc[-1], 3), " 富 0 の人数:", sum(1 for w in w1 if w == 0))


# 2
class HalfAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.wealth = 1

    def step(self):
        give = self.wealth // 2
        if give > 0:
            other = self.random.choice(list(self.model.agents))
            other.wealth += give
            self.wealth -= give


class HalfModel(mesa.Model):
    def __init__(self, n, seed=None):
        super().__init__(rng=seed)
        for _ in range(n):
            HalfAgent(self)
        self.datacollector = DataCollector(model_reporters={"Gini": compute_gini})

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")


m2 = HalfModel(100, seed=7)
for _ in range(200):
    m2.step()
df2 = m2.datacollector.get_model_vars_dataframe()
plt.plot(df1["Gini"], label="1 単位を渡す")
plt.plot(df2["Gini"], label="半分を渡す")
plt.xlabel("期")
plt.ylabel("ジニ係数")
plt.legend()
plt.show()
```

</details>

---
## 4. 空間のあるモデル：MultiGrid

現実の経済主体には「場所」があります。Mesa の **`MultiGrid`** は、1 つのマスに複数のエージェントが
入れる格子空間です。

| メソッド | 意味 |
|---|---|
| `MultiGrid(width, height, torus)` | 格子を作る。`torus=True` なら端がつながる（ドーナツ状） |
| `grid.place_agent(agent, (x, y))` | エージェントを置く（`agent.pos` に位置が入る） |
| `grid.move_agent(agent, (x, y))` | 移動する |
| `grid.get_neighborhood(pos, moore, include_center)` | 近傍のマスの座標一覧 |
| `grid.get_cell_list_contents([pos])` | マスにいるエージェント一覧 |
| `grid.get_neighbors(pos, moore, include_center)` | 近傍にいるエージェント一覧 |

`moore=True` は周囲 8 マス、`moore=False` は上下左右 4 マス（フォン・ノイマン近傍）です。

### 4.1 格子の基本操作

In [ ]:
class DotAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)


class GridDemoModel(mesa.Model):
    def __init__(self, seed=None):
        super().__init__(rng=seed)
        self.grid = MultiGrid(5, 5, torus=True)
        a = DotAgent(self)
        self.grid.place_agent(a, (2, 2))
        b = DotAgent(self)
        self.grid.place_agent(b, (2, 3))

    def step(self):
        pass


gm = GridDemoModel(seed=0)
a, b = gm.agents[0], gm.agents[1]
print("a の位置:", a.pos, " b の位置:", b.pos)
print("a の近傍マス（8 近傍）:", gm.grid.get_neighborhood(a.pos, moore=True, include_center=False))
print("a の近傍マス（4 近傍）:", gm.grid.get_neighborhood(a.pos, moore=False, include_center=False))
print("a の近傍にいるエージェント:", [n.unique_id for n in gm.grid.get_neighbors(a.pos, moore=True, include_center=False)])

gm.grid.move_agent(a, (0, 0))
print("移動後の a の位置:", a.pos)
print("(0,0) の中身:", [n.unique_id for n in gm.grid.get_cell_list_contents([(0, 0)])])

### 4.2 移動しながら取引する富のモデル

第 3 章のモデルに空間を加えます。エージェントは毎期ランダムに隣のマスへ移動し、
**同じマスにいる相手** にだけお金を渡します（出会わなければ取引できません）。

In [ ]:
class MovingMoneyAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.wealth = 1

    def move(self):
        candidates = self.model.grid.get_neighborhood(self.pos, moore=True, include_center=False)
        self.model.grid.move_agent(self, self.random.choice(candidates))

    def give_money(self):
        cellmates = [a for a in self.model.grid.get_cell_list_contents([self.pos]) if a is not self]
        if cellmates:
            other = self.random.choice(cellmates)
            other.wealth += 1
            self.wealth -= 1

    def step(self):
        self.move()
        if self.wealth > 0:
            self.give_money()


class MovingMoneyModel(mesa.Model):
    def __init__(self, n, width, height, seed=None):
        super().__init__(rng=seed)
        self.grid = MultiGrid(width, height, torus=True)
        for _ in range(n):
            a = MovingMoneyAgent(self)
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(a, pos)
        self.datacollector = DataCollector(model_reporters={"Gini": compute_gini})

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")


mm = MovingMoneyModel(50, 10, 10, seed=1)
for _ in range(100):
    mm.step()
print("最終期のジニ係数:", round(mm.datacollector.get_model_vars_dataframe()["Gini"].iloc[-1], 3))

### 4.3 格子上の分布を可視化する

各マスの **エージェント数** と **富の合計** を 2 次元配列にして `imshow` で描きます。

In [ ]:
counts = np.zeros((mm.grid.width, mm.grid.height))
wealth_map = np.zeros((mm.grid.width, mm.grid.height))
for contents, (x, y) in mm.grid.coord_iter():
    counts[x, y] = len(contents)
    wealth_map[x, y] = sum(a.wealth for a in contents)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(counts.T, origin="lower", cmap="Blues")
axes[0].set_title("マスごとのエージェント数")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(wealth_map.T, origin="lower", cmap="Oranges")
axes[1].set_title("マスごとの富の合計")
plt.colorbar(im1, ax=axes[1])
plt.show()

### 4.4 隣のマスにいる相手とも取引する：get_neighbors

「同じマス」だけでなく「周囲 8 マス」にいる相手とも取引できるようにすると、出会いの機会が増えます。
`get_neighbors(pos, moore=True, include_center=True)` で自分のマスを含む近傍のエージェントが取れます。

In [ ]:
class NeighborMoneyAgent(MovingMoneyAgent):
    def give_money(self):
        nearby = [a for a in self.model.grid.get_neighbors(self.pos, moore=True, include_center=True) if a is not self]
        if nearby:
            other = self.random.choice(nearby)
            other.wealth += 1
            self.wealth -= 1


class NeighborMoneyModel(MovingMoneyModel):
    def __init__(self, n, width, height, seed=None):
        mesa.Model.__init__(self, rng=seed)
        self.grid = MultiGrid(width, height, torus=True)
        for _ in range(n):
            a = NeighborMoneyAgent(self)
            self.grid.place_agent(a, (self.random.randrange(width), self.random.randrange(height)))
        self.datacollector = DataCollector(model_reporters={"Gini": compute_gini})

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")


nm = NeighborMoneyModel(50, 10, 10, seed=1)
for _ in range(100):
    nm.step()
print("同じマスのみ   :", round(mm.datacollector.get_model_vars_dataframe()["Gini"].iloc[-1], 3))
print("周囲 8 マスも含む:", round(nm.datacollector.get_model_vars_dataframe()["Gini"].iloc[-1], 3))

### 練習問題 3

1. `MovingMoneyModel` を 20×20 の格子・100 人・`seed=5` で 100 期動かし、ジニ係数の推移を第 3 章（空間なし、100 人・`seed=5`・100 期）と同じグラフに重ねて比べてください。
2. `move()` を「4 近傍（`moore=False`）だけに移動する」ように変えた `MovingMoneyAgent4` を作り、同じ条件で最終期のジニ係数を表示してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
m_space = MovingMoneyModel(100, 20, 20, seed=5)
m_nospace = MoneyModel2(100, seed=5)
for _ in range(100):
    m_space.step()
    m_nospace.step()
plt.plot(m_space.datacollector.get_model_vars_dataframe()["Gini"], label="空間あり（20×20）")
plt.plot(m_nospace.datacollector.get_model_vars_dataframe()["Gini"], label="空間なし")
plt.xlabel("期")
plt.ylabel("ジニ係数")
plt.legend()
plt.show()


# 2
class MovingMoneyAgent4(MovingMoneyAgent):
    def move(self):
        candidates = self.model.grid.get_neighborhood(self.pos, moore=False, include_center=False)
        self.model.grid.move_agent(self, self.random.choice(candidates))


class MovingMoneyModel4(mesa.Model):
    def __init__(self, n, width, height, seed=None):
        super().__init__(rng=seed)
        self.grid = MultiGrid(width, height, torus=True)
        for _ in range(n):
            a = MovingMoneyAgent4(self)
            self.grid.place_agent(a, (self.random.randrange(width), self.random.randrange(height)))
        self.datacollector = DataCollector(model_reporters={"Gini": compute_gini})

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")


m4 = MovingMoneyModel4(100, 20, 20, seed=5)
for _ in range(100):
    m4.step()
print("4 近傍のジニ係数:", round(m4.datacollector.get_model_vars_dataframe()["Gini"].iloc[-1], 3))
```

</details>

---
## 5. 買い手と売り手の市場モデル

需要と供給から均衡価格が決まる、というミクロ経済学の基本を ABM で再現してみます。

- **買い手**：それぞれ「支払ってもよい上限価格（支払意思額, WTP）」を持つ
- **売り手**：それぞれ「これ以上なら売る価格（費用）」と、現在の **提示価格** を持つ
- 毎期、買い手と売り手をランダムに組み合わせ、`提示価格 <= WTP` なら取引が成立する
- 売り手は、売れたら提示価格を少し **上げ**、売れ残ったら少し **下げる**

理論上の均衡価格は、WTP を高い順に並べた需要曲線と、費用を低い順に並べた供給曲線の交点です。

### 5.1 エージェントの定義

買い手と売り手は別のクラスにします。`model.agents_by_type[クラス]` で種類ごとの AgentSet が取れます。

In [ ]:
class Buyer(mesa.Agent):
    def __init__(self, model, wtp):
        super().__init__(model)
        self.wtp = wtp          # 支払意思額
        self.bought = False


class Seller(mesa.Agent):
    def __init__(self, model, cost):
        super().__init__(model)
        self.cost = cost        # 費用（これ未満では売らない）
        self.price = cost * 1.5 # 最初の提示価格は強気に
        self.sold = False

    def adjust_price(self):
        if self.sold:
            self.price *= 1.02                       # 売れたら 2% 値上げ
        else:
            self.price = max(self.cost, self.price * 0.97)   # 売れ残ったら 3% 値下げ（費用が下限）

### 5.2 市場モデル

`step()` の中で、買い手と売り手をシャッフルして順にマッチングします。
取引価格の平均と取引量を `DataCollector` に記録します。

In [ ]:
class MarketModel(mesa.Model):
    def __init__(self, n_buyers, n_sellers, seed=None):
        super().__init__(rng=seed)
        rng = self.random
        for _ in range(n_buyers):
            Buyer(self, wtp=rng.uniform(50, 150))
        for _ in range(n_sellers):
            Seller(self, cost=rng.uniform(30, 130))
        self.trade_prices = []
        self.datacollector = DataCollector(
            model_reporters={
                "平均取引価格": lambda m: np.mean(m.trade_prices) if m.trade_prices else np.nan,
                "取引量": lambda m: len(m.trade_prices),
            }
        )

    def step(self):
        buyers = list(self.agents_by_type[Buyer])
        sellers = list(self.agents_by_type[Seller])
        self.random.shuffle(buyers)
        self.random.shuffle(sellers)
        for b in buyers:
            b.bought = False
        for s in sellers:
            s.sold = False

        self.trade_prices = []
        for b, s in zip(buyers, sellers):          # ランダムに 1 対 1 で出会う
            if s.price <= b.wtp:
                self.trade_prices.append(s.price)
                b.bought = True
                s.sold = True
        self.datacollector.collect(self)
        for s in sellers:
            s.adjust_price()


market = MarketModel(n_buyers=60, n_sellers=60, seed=2)
for _ in range(100):
    market.step()

market_df = market.datacollector.get_model_vars_dataframe()
print(market_df.tail())

### 5.3 理論上の均衡価格と比べる

In [ ]:
wtps = sorted(market.agents_by_type[Buyer].get("wtp"), reverse=True)   # 需要曲線（高い順）
costs = sorted(market.agents_by_type[Seller].get("cost"))              # 供給曲線（低い順）

# 需要 >= 供給 が成り立つ最後の数量が均衡取引量
eq_q = sum(1 for w, c in zip(wtps, costs) if w >= c)
eq_p = (wtps[eq_q - 1] + costs[eq_q - 1]) / 2
print(f"理論上の均衡: 取引量 {eq_q}, 価格 {eq_p:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].step(range(1, len(wtps) + 1), wtps, label="需要（WTP）")
axes[0].step(range(1, len(costs) + 1), costs, label="供給（費用）")
axes[0].axhline(eq_p, color="gray", linestyle=":")
axes[0].set_xlabel("数量")
axes[0].set_ylabel("価格")
axes[0].set_title("需要曲線と供給曲線")
axes[0].legend()

axes[1].plot(market_df["平均取引価格"], label="シミュレーションの平均取引価格")
axes[1].axhline(eq_p, color="red", linestyle="--", label="理論上の均衡価格")
axes[1].set_xlabel("期")
axes[1].set_ylabel("価格")
axes[1].set_title("価格の推移")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
market_df["取引量"].plot()
plt.axhline(eq_q, color="red", linestyle="--", label="理論上の均衡取引量")
plt.title("取引量の推移")
plt.xlabel("期")
plt.ylabel("取引量")
plt.legend()
plt.show()

### 練習問題 4

1. 売り手を 60 人から 40 人に減らした市場（買い手 60 人、`seed=2`）を 100 期動かし、最終 10 期の平均取引価格を、売り手 60 人の場合と比べてください（供給が減ると価格はどうなるでしょうか）。
2. 売り手の値下げ幅を 3% から 10% に変えた `FastSeller` を作り（`adjust_price` を上書き）、価格が均衡に近づく速さを比べてください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
market40 = MarketModel(n_buyers=60, n_sellers=40, seed=2)
for _ in range(100):
    market40.step()
df40 = market40.datacollector.get_model_vars_dataframe()
print("売り手 60 人:", round(market_df["平均取引価格"].tail(10).mean(), 1))
print("売り手 40 人:", round(df40["平均取引価格"].tail(10).mean(), 1))


# 2
class FastSeller(Seller):
    def adjust_price(self):
        if self.sold:
            self.price *= 1.02
        else:
            self.price = max(self.cost, self.price * 0.90)


class FastMarketModel(MarketModel):
    def __init__(self, n_buyers, n_sellers, seed=None):
        mesa.Model.__init__(self, rng=seed)
        rng = self.random
        for _ in range(n_buyers):
            Buyer(self, wtp=rng.uniform(50, 150))
        for _ in range(n_sellers):
            FastSeller(self, cost=rng.uniform(30, 130))
        self.trade_prices = []
        self.datacollector = DataCollector(
            model_reporters={"平均取引価格": lambda m: np.mean(m.trade_prices) if m.trade_prices else np.nan}
        )

    def step(self):
        buyers = list(self.agents_by_type[Buyer])
        sellers = list(self.agents_by_type[FastSeller])
        self.random.shuffle(buyers)
        self.random.shuffle(sellers)
        for s in sellers:
            s.sold = False
        self.trade_prices = []
        for b, s in zip(buyers, sellers):
            if s.price <= b.wtp:
                self.trade_prices.append(s.price)
                s.sold = True
        self.datacollector.collect(self)
        for s in sellers:
            s.adjust_price()


fast = FastMarketModel(60, 60, seed=2)
for _ in range(100):
    fast.step()
plt.plot(market_df["平均取引価格"], label="値下げ 3%")
plt.plot(fast.datacollector.get_model_vars_dataframe()["平均取引価格"], label="値下げ 10%")
plt.xlabel("期")
plt.ylabel("平均取引価格")
plt.legend()
plt.show()
```

</details>

---
## 6. ネットワーク上の情報伝播（口コミ）

新製品の普及や流行は、人と人のつながり（ネットワーク）を通じて広がります。
ここでは NetworkX でつながりを作り、各エージェントを 1 つのノードに対応させます。

ルール（**閾値モデル**）：

- 採用していないエージェントは、隣人のうち採用者の割合が **閾値** 以上なら採用する
- さらに、広告などの効果として、毎期小さな確率 `p_ad` で自発的に採用する

### 6.1 モデルの定義

In [ ]:
class ConsumerAgent(mesa.Agent):
    def __init__(self, model, node, threshold):
        super().__init__(model)
        self.node = node               # 対応するネットワーク上のノード番号
        self.threshold = threshold
        self.adopted = False

    def step(self):
        if self.adopted:
            return
        if self.random.random() < self.model.p_ad:          # 広告で採用
            self.adopted = True
            return
        neighbors = list(self.model.G.neighbors(self.node))
        if not neighbors:
            return
        share = sum(self.model.node_agent[n].adopted for n in neighbors) / len(neighbors)
        if share >= self.threshold:                         # 口コミで採用
            self.adopted = True


class DiffusionModel(mesa.Model):
    def __init__(self, n, k=4, p_rewire=0.1, p_ad=0.01, threshold=0.3, n_seeds=2, seed=None):
        super().__init__(rng=seed)
        self.G = nx.watts_strogatz_graph(n, k, p_rewire, seed=seed)   # スモールワールド・ネットワーク
        self.p_ad = p_ad
        self.node_agent = {}
        for node in self.G.nodes:
            a = ConsumerAgent(self, node, threshold)
            self.node_agent[node] = a
        for node in self.random.sample(list(self.G.nodes), n_seeds):   # 最初の採用者
            self.node_agent[node].adopted = True
        self.datacollector = DataCollector(
            model_reporters={"採用率": lambda m: np.mean(m.agents.get("adopted"))}
        )

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")


diff = DiffusionModel(100, seed=1)
for _ in range(40):
    diff.step()
diff_df = diff.datacollector.get_model_vars_dataframe()
print(diff_df["採用率"].round(2).tolist()[:15])

### 6.2 普及曲線（S 字カーブ）

In [ ]:
diff_df["採用率"].plot(marker="o")
plt.title("新製品の普及率の推移（スモールワールド・ネットワーク）")
plt.xlabel("期")
plt.ylabel("採用率")
plt.ylim(0, 1.05)
plt.grid(True)
plt.show()

### 6.3 ネットワークの上に採用状況を描く

In [ ]:
pos = nx.circular_layout(diff.G)
colors = ["tab:red" if diff.node_agent[n].adopted else "lightgray" for n in diff.G.nodes]
plt.figure(figsize=(6, 6))
nx.draw(diff.G, pos, node_color=colors, node_size=60, edge_color="#cccccc")
plt.title("40 期後の採用状況（赤 = 採用）")
plt.show()

### 6.4 ネットワーク構造による違い

つながり方（ランダム、スモールワールド、スケールフリー）で普及の速さがどう変わるか比べます。

In [ ]:
class DiffusionModelG(DiffusionModel):
    """外から与えたグラフ G を使う版"""

    def __init__(self, G, p_ad=0.01, threshold=0.3, n_seeds=2, seed=None):
        mesa.Model.__init__(self, rng=seed)
        self.G = G
        self.p_ad = p_ad
        self.node_agent = {}
        for node in self.G.nodes:
            self.node_agent[node] = ConsumerAgent(self, node, threshold)
        for node in self.random.sample(list(self.G.nodes), n_seeds):
            self.node_agent[node].adopted = True
        self.datacollector = DataCollector(model_reporters={"採用率": lambda m: np.mean(m.agents.get("adopted"))})


graphs = {
    "ランダム（ER）": nx.gnm_random_graph(100, 200, seed=1),
    "スモールワールド（WS）": nx.watts_strogatz_graph(100, 4, 0.1, seed=1),
    "スケールフリー（BA）": nx.barabasi_albert_graph(100, 2, seed=1),
}
for name, G in graphs.items():
    m = DiffusionModelG(G, seed=1)
    for _ in range(40):
        m.step()
    m.datacollector.get_model_vars_dataframe()["採用率"].plot(label=name)
plt.title("ネットワーク構造と普及の速さ")
plt.xlabel("期")
plt.ylabel("採用率")
plt.legend()
plt.grid(True)
plt.show()

### 6.5 誰から広めるか：ハブ（つながりの多い人）から始める

最初の採用者を **ランダムに選ぶ** 場合と、**次数（つながりの数）が多い人を選ぶ** 場合で、普及の速さを比べます。
インフルエンサー・マーケティングの考え方です。

In [ ]:
class DiffusionModelHub(DiffusionModelG):
    """次数の大きいノードを最初の採用者にする版"""

    def __init__(self, G, p_ad=0.01, threshold=0.3, n_seeds=2, seed=None):
        super().__init__(G, p_ad=p_ad, threshold=threshold, n_seeds=0, seed=seed)   # まず採用者なしで作る
        hubs = sorted(G.degree, key=lambda x: x[1], reverse=True)[:n_seeds]
        for node, _ in hubs:
            self.node_agent[node].adopted = True


G_ba = nx.barabasi_albert_graph(100, 2, seed=2)
for label, cls in [("ランダムに選ぶ", DiffusionModelG), ("ハブから始める", DiffusionModelHub)]:
    m = cls(G_ba, p_ad=0.0, threshold=0.25, n_seeds=3, seed=2)
    for _ in range(30):
        m.step()
    m.datacollector.get_model_vars_dataframe()["採用率"].plot(label=label)
plt.title("最初の採用者の選び方と普及の速さ（広告なし、スケールフリー）")
plt.xlabel("期")
plt.ylabel("採用率")
plt.legend()
plt.grid(True)
plt.show()

### 練習問題 5

1. 閾値 `threshold` を 0.2、0.4、0.6 に変えて（`DiffusionModel(100, threshold=..., seed=1)`、40 期）、普及率の推移を 1 つのグラフに重ねてください。
2. 広告確率 `p_ad` を 0 にすると（口コミだけ）、40 期後の採用率はどうなりますか。`seed` を 1〜5 で変えて平均を出してください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
for th in [0.2, 0.4, 0.6]:
    m = DiffusionModel(100, threshold=th, seed=1)
    for _ in range(40):
        m.step()
    m.datacollector.get_model_vars_dataframe()["採用率"].plot(label=f"閾値 {th}")
plt.xlabel("期")
plt.ylabel("採用率")
plt.legend()
plt.show()

# 2
rates = []
for s in range(1, 6):
    m = DiffusionModel(100, p_ad=0.0, seed=s)
    for _ in range(40):
        m.step()
    rates.append(np.mean(m.agents.get("adopted")))
print("seed ごとの採用率:", [round(r, 2) for r in rates], " 平均:", round(np.mean(rates), 3))
```

</details>

---
## 7. パラメータ実験と再現性

ABM の結果は乱数に左右されるので、**同じ条件を複数の seed で繰り返し**、平均や散らばりを見るのが基本です。
また、**seed を固定すれば同じ結果を再現できる** ことを確認しておきましょう。

### 7.1 再現性の確認

In [ ]:
def run_money_model(n, steps, seed):
    m = MoneyModel2(n, seed=seed)
    for _ in range(steps):
        m.step()
    return m.datacollector.get_model_vars_dataframe()["Gini"].iloc[-1]


print("seed=1 の 1 回目:", round(run_money_model(50, 100, seed=1), 4))
print("seed=1 の 2 回目:", round(run_money_model(50, 100, seed=1), 4))
print("seed=2         :", round(run_money_model(50, 100, seed=2), 4))

### 7.2 パラメータを変えて繰り返す

エージェント数を変えながら、それぞれ 5 つの seed で実行し、結果を DataFrame にまとめます。

In [ ]:
records = []
for n in [10, 25, 50, 100]:
    for seed in range(5):
        records.append({"人数": n, "seed": seed, "ジニ係数": run_money_model(n, 100, seed)})
results = pd.DataFrame(records)
summary = results.groupby("人数")["ジニ係数"].agg(["mean", "std"]).round(3)
print(summary)

In [ ]:
plt.errorbar(summary.index, summary["mean"], yerr=summary["std"], marker="o", capsize=4)
plt.title("エージェント数とジニ係数（100 期後、5 回の平均 ± 標準偏差）")
plt.xlabel("エージェント数")
plt.ylabel("ジニ係数")
plt.grid(True)
plt.show()

### 7.3 期数を変える

同じモデルを長く動かすと、ジニ係数はどこかで落ち着く（定常状態）でしょうか。

In [ ]:
m = MoneyModel2(50, seed=0)
for _ in range(300):
    m.step()
g = m.datacollector.get_model_vars_dataframe()["Gini"]
g.rolling(20).mean().plot(label="20 期移動平均")
g.plot(alpha=0.3, label="各期の値")
plt.title("ジニ係数の長期的な推移（50 人・300 期）")
plt.xlabel("期")
plt.ylabel("ジニ係数")
plt.legend()
plt.show()

### 7.4 結果を保存する

実験結果は CSV に保存しておくと、あとで pandas や Excel で分析できます（JupyterLite では左のファイルブラウザに現れます）。

In [ ]:
results.to_csv("mesa_results.csv", index=False)
loaded = pd.read_csv("mesa_results.csv")
print(loaded.head())
print("保存した行数:", len(loaded))

### 練習問題 6

1. `MarketModel`（買い手 60 人）について、売り手の人数を 30, 45, 60, 75 と変え、それぞれ seed 0〜2 の 3 回ずつ 100 期動かして、最終 10 期の平均取引価格の平均を表にしてください。
2. その結果を、横軸＝売り手の人数、縦軸＝平均取引価格の折れ線グラフにしてください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
records = []
for n_sellers in [30, 45, 60, 75]:
    for seed in range(3):
        m = MarketModel(60, n_sellers, seed=seed)
        for _ in range(100):
            m.step()
        price = m.datacollector.get_model_vars_dataframe()["平均取引価格"].tail(10).mean()
        records.append({"売り手": n_sellers, "seed": seed, "価格": price})
res = pd.DataFrame(records)
tbl = res.groupby("売り手")["価格"].mean().round(1)
print(tbl)

# 2
tbl.plot(marker="o")
plt.xlabel("売り手の人数")
plt.ylabel("平均取引価格（最終 10 期）")
plt.grid(True)
plt.show()
```

</details>

---
## まとめ

| トピック | 主なクラス・メソッド |
|---|---|
| エージェント | `class A(mesa.Agent)`, `super().__init__(model)`, `self.unique_id`, `self.random`, `step()` |
| モデル | `class M(mesa.Model)`, `super().__init__(rng=...)`, `step()`, `model.steps` |
| AgentSet | `model.agents`, `shuffle_do("step")`, `get("attr")`, `select(条件)`, `agents_by_type[クラス]` |
| データ収集 | `DataCollector(model_reporters=..., agent_reporters=...)`, `collect()`, `get_model_vars_dataframe()`, `get_agent_vars_dataframe()` |
| 空間 | `MultiGrid`, `place_agent`, `move_agent`, `get_neighborhood`, `get_cell_list_contents`, `coord_iter` |
| ネットワーク | NetworkX のグラフ + `node → agent` の辞書 |
| 実験 | seed の固定、パラメータ × seed のループ、`groupby` で集計 |

## 次のステップ

- `python/simpy/simpy_beginner_tutorial.ipynb` — 待ち行列や在庫など「時間の流れ」を扱う離散事象シミュレーション
- `python/networkx/networkx_beginner_tutorial.ipynb` — ネットワーク分析の基礎（中心性、コミュニティ）
- `python/pyvis/pyvis_beginner_tutorial.ipynb` — ネットワークの対話的な可視化
- Mesa 公式のサンプル（Schelling の住み分けモデル、Sugarscape など）を Mesa 3 の書き方で自分で実装してみましょう

---
## 総合演習：所得税と再分配のシミュレーション

第 3 章の富の分配モデルに **税と再分配** を加えて、格差がどう変わるかを調べてください。

1. `TaxModel(n, tax_rate, seed)` を作る。毎期の流れは次のとおり：
   1. `DataCollector` でジニ係数を記録する
   2. 全エージェントが `MoneyAgent` と同じルールで 1 単位を渡す
   3. 各エージェントの富に `tax_rate` を掛けた額（切り捨て）を徴収し、合計を全員に均等に配る（余りは配らない）
2. 税率 0%、10%、30% で 100 人・200 期・`seed=1` を実行し、ジニ係数の推移を 1 つのグラフに重ねる。
3. 各税率について最終期の富の分布のヒストグラムを並べて描く。
4. 税率 0〜50%（10% 刻み）× seed 0〜2 で最終期ジニ係数の平均を表にする。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
class TaxModel(mesa.Model):
    def __init__(self, n, tax_rate, seed=None):
        super().__init__(rng=seed)
        self.tax_rate = tax_rate
        for _ in range(n):
            MoneyAgent(self)
        self.datacollector = DataCollector(model_reporters={"Gini": compute_gini})

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")
        # 徴税
        revenue = 0
        for a in self.agents:
            tax = int(a.wealth * self.tax_rate)
            a.wealth -= tax
            revenue += tax
        # 均等に再分配
        share = revenue // len(self.agents)
        for a in self.agents:
            a.wealth += share


def run_tax(tax_rate, seed, n=100, steps=200):
    m = TaxModel(n, tax_rate, seed=seed)
    for _ in range(steps):
        m.step()
    return m


# 2. 税率別のジニ係数の推移
models = {rate: run_tax(rate, seed=1) for rate in [0.0, 0.1, 0.3]}
for rate, m in models.items():
    m.datacollector.get_model_vars_dataframe()["Gini"].plot(label=f"税率 {int(rate * 100)}%")
plt.title("税率とジニ係数の推移（100 人・200 期）")
plt.xlabel("期")
plt.ylabel("ジニ係数")
plt.legend()
plt.grid(True)
plt.show()

# 3. 最終期の分布
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, (rate, m) in zip(axes, models.items()):
    w = m.agents.get("wealth")
    ax.hist(w, bins=range(0, max(w) + 2), edgecolor="black")
    ax.set_title(f"税率 {int(rate * 100)}%")
    ax.set_xlabel("富")
axes[0].set_ylabel("人数")
plt.tight_layout()
plt.show()

# 4. 税率 × seed の表
records = []
for rate in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    for seed in range(3):
        m = run_tax(rate, seed)
        records.append({"税率": rate, "seed": seed, "Gini": m.datacollector.get_model_vars_dataframe()["Gini"].iloc[-1]})
print(pd.DataFrame(records).groupby("税率")["Gini"].mean().round(3))

お疲れさまでした！ ABM は「ルールは単純なのに、全体の振る舞いは複雑」という現象を体験するのに最適な道具です。
自分の関心のある経済現象（バブル、企業の参入退出、感染症と経済活動など）をモデルにしてみましょう。